In [1]:
import json, shutil, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
threadpool_limits(limits=1)

In [2]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'

In [3]:
config = json.loads((INPUT / 'config.json').read_text())
SEED = config['seed']
rng = np.random.default_rng(SEED)

In [4]:
for cell in json.loads((INPUT / 'helpers.ipynb').read_text())['cells']:
    exec(''.join(cell['source']), globals())
carry(INPUT, OUTPUT)

In [5]:
data = np.load(INPUT / 'digits.npz')
x, y, xt, yt = (data[k] for k in ['x_train', 'y_train', 'x_test', 'y_test'])
nets = load_nets(INPUT / 'trained_networks.npz')
root = nets['root']

In [6]:
specifications = []
for split, long_n, short_n, single_n, missing_n in [
    ('cal', config['calibration_long'], config['calibration_short'], 0, 0),
    ('test', config['test_long'], config['test_short'], config['test_single'], config['test_missing'])]:
    for kind, n, mean, sources in [('long', long_n, config['mean_long_tract'], 3),
                                  ('short', short_n, config['mean_short_tract'], 3),
                                  ('single', single_n, 24, 3), ('missing', missing_n, config['mean_long_tract'], 4)]:
        for j in range(n):
            path = ancestry_path(rng, config['blocks'], sources, mean, 'single' if kind == 'single' else 'mosaic')
            # every missing-source case must actually contain that source
            if kind == 'missing':
                path[8:12] = 3
            specifications.append({'mosaic_id': f'{split}_{kind}_{j:03d}', 'split': split,
                                   'kind': kind, 'path': path})

In [7]:
vectors, labels, records, checkpoint_nets, permutation_rows = [], [], [], {}, []
permutation = rng.permutation(config['width'])
inverse = np.argsort(permutation)
for specification in specifications:
    split, mosaic_id, kind, path = (specification[k] for k in ['split', 'mosaic_id', 'kind', 'path'])
    donors = [nets[f'{split}_donor{k}'] for k in range(config['all_sources'])]
    net = splice(root, donors, path)
    # ground truth comes from the actual copied tensors
    assert all(np.array_equal(net['block_w'][j], donors[k]['block_w'][j]) and
               np.array_equal(net['block_b'][j], donors[k]['block_b'][j]) for j, k in enumerate(path))
    optimizer, previous = None, 0
    for steps in config['finetune_steps']:
        optimizer = train(net, x, y, steps - previous, rng, config['finetune_lr'], config['batch'], optimizer=optimizer)
        previous = steps
        query_id = len(records)
        vectors.append(block_vectors(net))
        labels.append(path.copy())
        records.append({'query_id': query_id, 'mosaic_id': mosaic_id, 'split': split, 'kind': kind,
                        'steps': steps, 'variant': 'aligned', **digit_metrics(net, xt, yt)})
        if split == 'test' and kind == 'long' and int(mosaic_id[-3:]) < config['permuted_mosaics'] and steps in [0, 300]:
            checkpoint_nets[f'{mosaic_id}_s{steps}'] = clone(net)
            permuted = permute(net, permutation)
            restored = permute(permuted, inverse)
            error = np.max(np.abs(forward(permuted, xt[:40]) - forward(net, xt[:40])))
            assert error < 1e-10
            assert all(np.array_equal(restored[key], net[key]) for key in net)
            permutation_rows.append({'mosaic_id': mosaic_id, 'steps': steps, 'logit_max_error': error})
            for variant, altered in [('permuted', permuted), ('oracle_restored', restored)]:
                vectors.append(block_vectors(altered))
                labels.append(path.copy())
                records.append({'query_id': len(records), 'mosaic_id': mosaic_id, 'split': split, 'kind': kind,
                                'steps': steps, 'variant': variant, **digit_metrics(altered, xt, yt)})
np.savez_compressed(OUTPUT / 'query_blocks.npz', blocks=np.stack(vectors))
np.savez_compressed(OUTPUT / 'truth.npz', ancestry=np.stack(labels))
pd.DataFrame(records).to_csv(OUTPUT / 'queries.csv', index=False)
pd.DataFrame(permutation_rows).to_csv(OUTPUT / 'permutation_checks.csv', index=False)
save_nets(OUTPUT / 'example_mosaics.npz', checkpoint_nets)

In [8]:
queries = pd.DataFrame(records)
queries.query("variant == 'aligned'").groupby(['split', 'kind', 'steps']).agg(
    mosaics=('mosaic_id', 'nunique'), digit_accuracy=('digit_accuracy', 'mean'))

mosaics  digit_accuracy
split kind    steps                         
cal   long    0           12        0.971111
              25          12        0.969815
              100         12        0.973333
              300         12        0.975000
      short   0            6        0.974074
              25           6        0.971852
              100          6        0.972963
              300          6        0.974074
test  long    0           24        0.973241
              25          24        0.970648
              100         24        0.973519
              300         24        0.973241
      missing 0           12        0.972593
              25          12        0.970556
              100         12        0.972963
              300         12        0.970000
      short   0           12        0.972593
              25          12        0.969259
              100         12        0.973519
              300         12        0.973333
      single  0            8        0.970278
              25           8        0.970000
              100          8        0.974444
              300          8        0.974444